Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/HNSCC_P_PDC000222"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                    Gene     Sequence                       Glycan                            
ENSP00000226382@158                     PHOX2B   AAAAAAAAAKNGSSGKK              N4H5F1S2G0            13.333897   
ENSP00000364585@5                       RCC2     AAAAAWEEPSSGNGTARAGPR          N5H7F2S2G0            12.326392   
ENSP00000349437@2122                    IGF2R    AACAVKPQEVQMVNGTITNPINGK       N4H4F0S2G0            14.687286   
                                                                                N4H5F1S1G0            15.421282   
ENSP00000353032@196;ENSP00000336607@180 P2RX4    AAENFTLLVK                     N3H6F0S1G0            14.218835   
...                                                                                                         ...   
ENSP00000376793@267;ENSP00000451119@49  SERPINA3 YTGNASALFILPDQDK               N8H9F1S0G0            13.271808   
                                                                                N9H6F1S0G0            12.386795   
                                                 YTGNASALFILPDQDKMEEVEAMLLPETLK N8H9F1S2G0            12.094518   
ENSP00000429273@32;ENSP00000478900@32   PCDHGB1  YTIPEELANGSRVGK                N7H6F0S1G0            13.731829   
ENSP00000322419@2                       RPLP2    YVASYLLAALGGNSSPSAK            N6H5F3S2G0            13.792819   

                                                                                              pool_01  \
Site                                    Gene     Sequence                       Glycan                  
ENSP00000226382@158                     PHOX2B   AAAAAAAAAKNGSSGKK              N4H5F1S2G0  13.333897   
ENSP00000364585@5                       RCC2     AAAAAWEEPSSGNGTARAGPR          N5H7F2S2G0  12.326392   
ENSP00000349437@2122                    IGF2R    AACAVKPQEVQMVNGTITNPINGK       N4H4F0S2G0  14.687286   
                                                                                N4H5F1S1G0  15.421282   
ENSP00000353032@196;ENSP00000336607@180 P2RX4    AAENFTLLVK                     N3H6F0S1G0  14.218835   
...                                                                                               ...   
ENSP00000376793@267;ENSP00000451119@49  SERPINA3 YTGNASALFILPDQDK               N8H9F1S0G0        NaN   
                                                                                N9H6F1S0G0        NaN   
                                                 YTGNASALFILPDQDKMEEVEAMLLPETLK N8H9F1S2G0        NaN   
ENSP00000429273@32;ENSP00000478900@32   PCDHGB1  YTIPEELANGSRVGK                N7H6F0S1G0        NaN   
ENSP00000322419@2                       RPLP2    YVASYLLAALGGNSSPSAK            N6H5F3S2G0        NaN   

                                                                                            C3L-00997_T_01  \
Site                                    Gene     Sequence                       Glycan                       
ENSP00000226382@158                     PHOX2B   AAAAAAAAAKNGSSGKK              N4H5F1S2G0       13.525016   
ENSP00000364585@5                       RCC2     AAAAAWEEPSSGNGTARAGPR          N5H7F2S2G0       11.724626   
ENSP00000349437@2122                    IGF2R    AACAVKPQEVQMVNGTITNPINGK       N4H4F0S2G0       14.689635   
                                                                                N4H5F1S1G0       15.670019   
ENSP00000353032@196;ENSP00000336607@180 P2RX4    AAENFTLLVK                     N3H6F0S1G0       13.403411   
...                                                                                                    ...   
ENSP00000376793@267;ENSP00000451119@49  SERPINA3 YTGNASALFILPDQDK               N8H9F1S0G0             NaN   
                                                                                N9H6F1S0G0             NaN   
                                                 YTGNASALFILPDQDKMEEVEAMLLPETLK N8H9F1S2G0             NaN   
ENSP00000429273@32;E

In [4]:
meta_path= os.path.join(meta_dir, "HNSCC_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,CASP8_mutation,FAT1_mutation,ARID1A_mutation,PIK3CA_mutation,TP53_mutation,CDKN2A_mutation,NOTCH1_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN,BIN,BIN,BIN
0,C3L-00977,56,Male,1.2,G1 Well differentiated,Not identified,pT1,pN1,Stage III,33.21,past smoker,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,C3L-00987,61,Male,4.0,G2 Moderately differentiated,Present,pT2,pN1,Stage III,24.15,current smoker,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,C3L-00994,50,Male,3.0,G2 Moderately differentiated,Present,pT2,pN0,Stage II,21.74,past smoker,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,C3L-00995,56,Male,4.0,G1 Well differentiated,Not identified,pT2,pN1,Stage III,20.76,past smoker,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,C3L-00997,47,Male,4.0,G2 Moderately differentiated,Present,pT2,pN1,Stage II,21.83,past smoker,0.0,0.0,0.0,0.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,C3N-04277,72,Male,1.0,G2 Moderately differentiated,Not identified,pT3,NaN,Stage III,26.00,current smoker,0.0,1.0,0.0,0.0,1.0,0.0,0.0
104,C3N-04278,71,Male,4.0,G2 Moderately differentiated,Not identified,pT3,NaN,Stage III,30.00,past smoker,0.0,1.0,0.0,0.0,1.0,1.0,0.0
105,C3N-04279,65,Male,3.0,G2 Moderately differentiated,Not identified,pT2,pN0,Stage II,22.00,current smoker,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,C3L-00977,Male,Stage III
1,C3L-00987,Male,Stage III
2,C3L-00994,Male,Stage II
3,C3L-00995,Male,Stage III
4,C3L-00997,Male,Stage II
...,...,...,...
103,C3N-04277,Male,Stage III
104,C3N-04278,Male,Stage III
105,C3N-04279,Male,Stage II
106,C3N-04280,Male,Stage II


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,C3L-00977,Male,Stage III
1,C3L-00987,Male,Stage III
2,C3L-00994,Male,Stage II
3,C3L-00995,Male,Stage III
4,C3L-00997,Male,Stage II
5,C3L-00999,Male,Stage II
6,C3L-01138,Male,Stage IV
7,C3L-01237,Male,Stage IV
8,C3L-02617,Male,Stage II
9,C3L-02621,Male,Stage I


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

110

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
4,C3L-00997_T_01,Male,Stage II
92,C3N-03849_T_01,Male,Stage IV
79,C3N-03487_T_01,Female,Stage IV
44,C3N-01858_T_01,Male,Stage IV
14,C3L-04791_T_01,Male,Stage I
100,C3N-04273_T_01,Male,Stage IV
85,C3N-03664_T_02,Male,Stage IV
86,C3N-03781_T_02,Male,Stage IV
5,C3L-00999_T_02,Male,Stage II
45,C3N-01859_T_02,Male,Stage IV


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,pool_01,C3L-00997_T_01,C3N-03849_T_01,C3N-01858_N_01,QC1_C_01,C3N-03487_T_01,C3L-00997_N_01,C3N-01858_T_01,C3L-04791_T_01,...,C3L-00994_N_R1_20,C3N-03042_N_20,C3L-02617_N_R1_20,QC9_C_20,C3L-00994_N_R2_20,C3N-01757_N_20,C3L-02617_N_R2_20,C3L-04350_N_20,C3L-05257_N_20,C3L-02617_T_20
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000226382@158,PHOX2B,AAAAAAAAAKNGSSGKK,N4H5F1S2G0,13.333897,13.333897,13.525016,13.287321,12.725623,13.652612,12.667584,12.857535,12.699111,13.502198,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000364585@5,RCC2,AAAAAWEEPSSGNGTARAGPR,N5H7F2S2G0,12.326392,12.326392,11.724626,11.810661,12.584516,12.873049,11.495011,12.618791,11.188609,12.372281,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
data_df.shape

(80619, 221)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
len(samples)

110

In [17]:
df2 = data_df.loc[:,samples].dropna()

In [18]:
df2.shape

(962, 110)

In [19]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [20]:
data2.shape

(853, 110)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [21]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [22]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [23]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N2H...,HM
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N2H...,HM
2,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N3H...,F+S
3,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,F+S
...,...,...
848,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
849,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
850,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
851,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S


Map glycan types with colors.

In [24]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [25]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [26]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)